In [22]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import MinMaxScaler

In [23]:
df = pd.read_csv(r"E:\SGP Fake Review\Dataset\reviewer_dataset.csv")

In [24]:
df.columns

Index(['reviewerID', 'avg_rating', 'total_reviews', 'rating_variance',
       'rating_entropy', 'first_review', 'last_review', 'active_days',
       'reviews/day', 'review_burst_7d', 'burst_ratio_7d', '1_star_reviews',
       '%_1Star', '2_star_reviews', '%_2Star', '3_star_reviews', '%_3Star',
       '4_star_reviews', '%_4Star', '5_star_reviews', '%_5Star',
       'unique_products_reviewed', 'total_helpful_votes',
       '%_helpful_per_review', 'avg_days_between_reviews'],
      dtype='object')

In [25]:
df_model = df[df['total_reviews']>=3]
# df_model.to_csv('df_model.csv')

In [26]:
df_model_02 = df_model

In [27]:
drop_cols = ["reviewerID",
             "first_review",
             "last_review",
             "1_star_reviews",
             "2_star_reviews",
             "3_star_reviews",
             "4_star_reviews",
             "5_star_reviews",
             "%_1Star",
             "%_2Star",
             "%_3Star",
             "%_4Star",
             "%_5Star",
             "unique_products_reviewed",
             "total_reviews",
             "active_days",
             "avg_rating",
             "%_helpful_per_review",
             "total_helpful_votes"]
df_model_02 = df_model_02.drop(columns = drop_cols)

In [28]:
df_model_02.reset_index(inplace = True)
df_model_02.drop(columns = 'index', inplace = True)
df_model_02

,rating_variance,rating_entropy,reviews/day,review_burst_7d,burst_ratio_7d,avg_days_between_reviews
0,1.200000,1.370951,0.014164,3,0.600000,88.000000
1,0.000000,0.000000,0.294118,3,0.600000,4.000000
2,0.000000,0.000000,0.035354,2,0.285714,32.833333
3,0.200000,0.721928,0.007874,1,0.200000,158.500000
4,1.604040,0.979971,0.022310,3,0.066667,45.818182
...,...,...,...,...,...,...
9430,2.785714,1.905639,0.002974,1,0.125000,383.571429
9431,0.000000,0.000000,0.004525,1,0.333333,330.500000
9432,4.000000,1.392147,0.009626,3,0.333333,116.500000
9433,0.285714,0.371232,0.005863,4,0.285714,183.307692


In [29]:
df_model_02['burst_intensity'] = df_model_02['review_burst_7d'] * df_model_02['burst_ratio_7d']

In [30]:
drop_cols = ["reviewerID",
             "first_review",
             "last_review",
             "1_star_reviews",
             "2_star_reviews",
             "3_star_reviews",
             "4_star_reviews",
             "5_star_reviews",
             "%_1Star",
             "%_2Star",
             "%_3Star",
             "%_4Star",
             "%_5Star",
             "unique_products_reviewed",
             "total_reviews",
             "active_days",
             "avg_rating",
             "avg_days_between_reviews",
             "%_helpful_per_review",
             "total_helpful_votes"]
df_model = df_model.drop(columns = drop_cols)

In [31]:
df_model.columns 

Index(['rating_variance', 'rating_entropy', 'reviews/day', 'review_burst_7d',
       'burst_ratio_7d'],
      dtype='object')

In [32]:
df_model.reset_index(inplace = True)

In [33]:
df_model.drop(columns = 'index', inplace = True)

In [34]:
df_model

,rating_variance,rating_entropy,reviews/day,review_burst_7d,burst_ratio_7d
0,1.200000,1.370951,0.014164,3,0.600000
1,0.000000,0.000000,0.294118,3,0.600000
2,0.000000,0.000000,0.035354,2,0.285714
3,0.200000,0.721928,0.007874,1,0.200000
4,1.604040,0.979971,0.022310,3,0.066667
...,...,...,...,...,...
9430,2.785714,1.905639,0.002974,1,0.125000
9431,0.000000,0.000000,0.004525,1,0.333333
9432,4.000000,1.392147,0.009626,3,0.333333
9433,0.285714,0.371232,0.005863,4,0.285714


In [35]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_model)

In [36]:
iso = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42
)

_ = iso.fit(X_scaled)
print("Isolation Forest trained successfully.")

Isolation Forest trained successfully.


In [37]:
lof = LocalOutlierFactor(
    n_neighbors=100,
    contamination=0.02
)

pred_lof = lof.fit_predict(X_scaled)

In [38]:
ocsvm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.01
)

pred_svm = ocsvm.fit_predict(X_scaled)

In [39]:
pred_iso = iso.predict(X_scaled)

In [40]:
scores_iso = iso.decision_function(X_scaled)

In [41]:

score_scaler = MinMaxScaler((0,100))

behavior_score_iso = score_scaler.fit_transform(
    scores_iso.reshape(-1,1)
)

In [42]:
df_model["prediction"] = pred_iso

df_model["decision_score"] = scores_iso

df_model["behavior_score"] = behavior_score_iso

In [43]:
anomalies = df_model[df_model["prediction"] == -1]

anomalies.describe().T

,count,mean,std,min,25%,50%,75%,max
rating_variance,188.0,1.395259,1.483091,0.000000,0.267560,0.835897,1.900000,5.333333
rating_entropy,188.0,0.957463,0.568754,0.000000,0.650022,0.918296,1.451399,2.251629
reviews/day,188.0,4.318865,3.612241,0.010846,1.575000,5.000000,6.000000,30.000000
review_burst_7d,188.0,7.617021,6.190568,2.000000,5.000000,6.000000,8.000000,43.000000
burst_ratio_7d,188.0,0.912625,0.231488,0.049430,1.000000,1.000000,1.000000,1.000000
prediction,188.0,-1.000000,0.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
decision_score,188.0,-0.039162,0.029168,-0.133574,-0.060574,-0.033457,-0.014730,-0.000340
behavior_score,188.0,23.854384,7.369701,0.000000,18.444278,25.295799,30.027520,33.663124


In [44]:
normals = df_model[df_model["prediction"] == 1]

normals.describe().T

,count,mean,std,min,25%,50%,75%,max
rating_variance,9247.0,0.896897,1.123514,0.000000,0.142857,0.467930,1.230769,5.333333
rating_entropy,9247.0,0.891292,0.599343,0.000000,0.503258,0.918296,1.370951,2.321928
reviews/day,9247.0,0.086421,0.502772,0.001028,0.004450,0.007865,0.014972,5.000000
review_burst_7d,9247.0,2.407592,1.744829,1.000000,1.000000,2.000000,3.000000,39.000000
burst_ratio_7d,9247.0,0.334336,0.208712,0.017660,0.176471,0.285714,0.400000,1.000000
prediction,9247.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
decision_score,9247.0,0.195182,0.054547,0.000000,0.174327,0.213767,0.232582,0.262211
behavior_score,9247.0,83.064471,13.781971,33.749150,77.795032,87.760068,92.513834,100.000000


In [45]:
normals[normals['review_burst_7d'] == 39]

,rating_variance,rating_entropy,reviews/day,review_burst_7d,burst_ratio_7d,prediction,decision_score,behavior_score
5673,0.355346,0.945691,0.026163,39,0.722222,1,0.011338,36.613858


In [46]:
df[df['review_burst_7d'] == 39]

,reviewerID,avg_rating,total_reviews,rating_variance,rating_entropy,first_review,last_review,active_days,reviews/day,review_burst_7d,...,3_star_reviews,%_3Star,4_star_reviews,%_4Star,5_star_reviews,%_5Star,unique_products_reviewed,total_helpful_votes,%_helpful_per_review,avg_days_between_reviews
6880,AEPMOBDBHD5X4IJGF7LF7J3TAJVQ,4.722222,54,0.355346,0.945691,2017-06-02 04:58:20.000,2023-01-25 07:26:47.711,2064,0.026163,39,...,1,1.851852,10,18.518519,42,77.777778,54,37,0.685185,38.886792
12765,AGPOLGAZX27H4VIFZIQKKTPYYDIA,4.782609,46,0.707246,0.513185,2016-01-02 07:44:24.000,2019-07-08 09:26:01.236,1284,0.035826,39,...,0,0.000000,2,4.347826,42,91.304348,46,35,0.760870,28.466667


In [47]:
comparison = pd.DataFrame({
    "IsolationForest": pred_iso,
    "LOF": pred_lof,
    "OneClassSVM": pred_svm
})

comparison.head()

,IsolationForest,LOF,OneClassSVM
0,1,1,1
1,1,1,1
2,1,1,1
3,1,1,1
4,1,1,1


In [48]:
for col in comparison.columns:
    print(f"{col}: {(comparison[col] == -1).sum()} anomalies")

IsolationForest: 188 anomalies
LOF: 189 anomalies
OneClassSVM: 130 anomalies


In [49]:
comparison["Consensus"] = (
    (comparison["IsolationForest"] == -1) &
    (comparison["LOF"] == -1) &
    (comparison["OneClassSVM"] == -1)
)

comparison["Consensus"].sum()

np.int64(4)

In [50]:
index = comparison[comparison["Consensus"] == True].index

In [51]:
df_model.loc[index]

,rating_variance,rating_entropy,reviews/day,review_burst_7d,burst_ratio_7d,prediction,decision_score,behavior_score
4151,0.562500,0.337290,16.000000,16,1.000000,-1,-0.116098,4.415634
5582,0.000000,0.000000,30.000000,30,1.000000,-1,-0.126793,1.713419
8288,0.172818,0.421280,0.235551,43,0.061341,-1,-0.075947,14.560245
9412,1.742647,1.159555,17.000000,17,1.000000,-1,-0.133574,0.000000


In [52]:
scaler = StandardScaler()
X_scaled_02 = scaler.fit_transform(df_model_02)

In [53]:
df_model_02["avg_days_between_reviews"] = np.log1p(df_model_02["avg_days_between_reviews"])


In [54]:

scaler = StandardScaler()
X_scaled_02 = scaler.fit_transform(df_model_02)
iso_02 = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42
)

_ = iso_02.fit(X_scaled_02)
print("Isolation Forest trained successfully.")

Isolation Forest trained successfully.


In [55]:
lof_02 = LocalOutlierFactor(
    n_neighbors=100,
    contamination=0.02
)

pred_lof_02 = lof_02.fit_predict(X_scaled_02)

In [56]:
ocsvm_02 = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.01
)

pred_svm_02 = ocsvm_02.fit_predict(X_scaled_02)

In [57]:
pred_iso_02 = iso_02.predict(X_scaled_02)

In [58]:

scores_iso_02 = iso_02.decision_function(X_scaled_02)
score_scaler = MinMaxScaler((0,100))

behavior_score_iso_02 = score_scaler.fit_transform(
    scores_iso_02.reshape(-1,1)
)

In [59]:
df_model_02["prediction"] = pred_iso_02

df_model_02["decision_score"] = scores_iso_02

df_model_02["behavior_score"] = behavior_score_iso_02

In [60]:
comparison_02 = pd.DataFrame({
    "IsolationForest": pred_iso_02,
    "LOF": pred_lof_02,
    "OneClassSVM": pred_svm_02
})

comparison_02.head()

,IsolationForest,LOF,OneClassSVM
0,1,1,1
1,1,1,1
2,1,1,1
3,1,1,1
4,1,-1,1


In [61]:
for col in comparison_02.columns:
    print(f"{col}: {(comparison_02[col] == -1).sum()} anomalies")

IsolationForest: 189 anomalies
LOF: 189 anomalies
OneClassSVM: 173 anomalies


In [62]:
comparison_02["Consensus"] = (
    (comparison_02["IsolationForest"] == -1) &
    (comparison_02["LOF"] == -1) &
    (comparison_02["OneClassSVM"] == -1)
)

comparison_02["Consensus"].sum()

np.int64(49)

In [63]:
index_02 = comparison_02[comparison_02["Consensus"] == True].index

In [64]:
df_model_02.loc[index_02]

,rating_variance,rating_entropy,reviews/day,review_burst_7d,burst_ratio_7d,avg_days_between_reviews,burst_intensity,prediction,decision_score,behavior_score
717,2.666667,2.251629,0.206897,5,0.833333,1.887070,4.166667,-1,-0.012911,29.842994
773,0.654545,0.684038,11.000000,11,1.000000,0.000000,11.000000,-1,-0.112202,5.517878
1751,4.444444,1.000000,0.080645,7,0.700000,2.685577,4.900000,-1,-0.010831,30.352552
1908,0.000000,0.000000,2.666667,16,1.000000,0.287682,16.000000,-1,-0.084645,12.268967
2589,2.000000,1.685475,10.000000,10,1.000000,0.000000,10.000000,-1,-0.123156,2.834211
2620,2.809524,1.664498,7.000000,7,1.000000,0.000000,7.000000,-1,-0.100250,8.445816
3080,1.714286,1.561278,8.000000,8,1.000000,0.000000,8.000000,-1,-0.101864,8.050439
3256,0.418182,0.865857,1.833333,11,1.000000,0.405465,11.000000,-1,-0.074472,14.761325
3433,0.000000,0.000000,1.428571,10,1.000000,0.510826,10.000000,-1,-0.067882,16.375777
3592,4.200000,1.521928,5.000000,5,1.000000,0.000000,5.000000,-1,-0.065202,17.032224


In [65]:
iso_len = set(np.where(pred_iso == -1)[0])
lof_len = set(np.where(pred_lof == -1)[0])
svm_len = set(np.where(pred_svm == -1)[0])

print("ISO ∩ LOF:", len(iso_len & lof_len))
print("ISO ∩ SVM:", len(iso_len & svm_len))
print("LOF ∩ SVM:", len(lof_len & svm_len))

print("All three:", len(iso_len & lof_len & svm_len))

ISO ∩ LOF: 4
ISO ∩ SVM: 69
LOF ∩ SVM: 5
All three: 4


In [66]:
iso_len_02 = set(np.where(pred_iso_02 == -1)[0])
lof_len_02 = set(np.where(pred_lof_02 == -1)[0])
svm_len_02 = set(np.where(pred_svm_02 == -1)[0])

print("ISO ∩ LOF:", len(iso_len_02 & lof_len_02))
print("ISO ∩ SVM:", len(iso_len_02 & svm_len_02))
print("LOF ∩ SVM:", len(lof_len_02 & svm_len_02))

print("All three:", len(iso_len_02 & lof_len_02 & svm_len_02))

ISO ∩ LOF: 83
ISO ∩ SVM: 65
LOF ∩ SVM: 77
All three: 49


In [3]:
# df_model.to_csv("df_model.csv", index = False)
# df_model_02.to_csv("df_model_02.csv", index = False)

In [2]:
# df_model[df_model['prediction'] == -1].to_csv('anomaly_01.csv', index = False)
# df_model_02[df_model_02['prediction'] == -1].to_csv('anomaly_02.csv', index = False)

In [69]:
df_model_02[(df_model_02['review_burst_7d']> 10) & (df_model_02['review_burst_7d']<40)].tail(43)

,rating_variance,rating_entropy,reviews/day,review_burst_7d,burst_ratio_7d,avg_days_between_reviews,burst_intensity,prediction,decision_score,behavior_score
6560,0.145423,0.495410,0.144470,17,0.132812,2.039275,2.257812,1,0.001607,33.399648
6811,0.296668,0.838008,0.053112,12,0.076433,2.979577,0.917197,1,0.058216,47.268294
6819,1.435897,1.145110,0.017196,11,0.846154,4.157580,9.307692,-1,-0.009726,30.623136
6984,2.411778,1.484597,0.225966,14,0.061404,1.618210,0.859649,-1,-0.024376,27.034229
7035,1.098086,1.983849,0.121260,31,0.402597,2.217486,12.480519,-1,-0.078131,13.864680
7192,0.433304,0.901652,0.073803,19,0.094059,2.661893,1.787129,1,0.014244,36.495537
7252,0.463673,1.091216,0.018149,28,0.560000,4.045554,15.680000,-1,-0.024677,26.960389
7281,0.000000,0.000000,0.047468,12,0.800000,3.157000,9.600000,-1,-0.024161,27.086725
7296,0.358923,1.167420,0.023494,20,0.363636,3.788390,7.272727,1,0.007501,34.843559
7333,0.000000,0.000000,0.078652,11,0.785714,2.682075,8.642857,-1,-0.016388,28.991069


In [70]:
df_model_02.drop(columns = ['prediction', 'decision_score', 'behavior_score'], inplace = True)

In [71]:
feature_names = [
    "rating_variance",
    "rating_entropy",
    "reviews/day",
    "review_burst_7d",
    "burst_ratio_7d",
    "avg_days_between_reviews",
    "burst_intensity"
]

X_weighted = pd.DataFrame(X_scaled_02, columns=feature_names)

# X_weighted["review_burst_7d"] *= 2
# X_weighted["burst_ratio_7d"] *= 1.5

_ = iso_02.fit(X_weighted)

In [72]:
pred_iso_02 = iso_02.predict(X_weighted)
scores_iso_02 = iso_02.decision_function(X_weighted)
score_scaler = MinMaxScaler((0,100))

behavior_score_iso_02 = score_scaler.fit_transform(
    scores_iso_02.reshape(-1,1)
)

In [73]:
df_model_02["prediction"] = pred_iso_02

df_model_02["decision_score"] = scores_iso_02

df_model_02["behavior_score"] = behavior_score_iso_02

In [1]:
# df_model_02[df_model_02['prediction'] == -1].to_csv('anomaly.csv')
# df_model_02[df_model_02['prediction'] == 1].to_csv('normal.csv')
